# 04 — Final Predictions
**DataStorm v7.0 | SkyNet Team**

Goal: Generate the final submission CSV with `Outlet_ID` and `Total_Latent_Demand` for January 2026.

In [3]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

GOLD_DIR = ROOT / 'data' / 'gold'
OUTPUTS  = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

print('Setup complete')

Setup complete


In [4]:
# Load potential demand results
results = pd.read_csv(GOLD_DIR / 'potential_demand_results.csv')
print('Results loaded:', results.shape)

# Select and rename columns for submission
# The target is 'Total_Latent_Demand' for January 2026
submission = results[['Outlet_ID', 'jan_2026_forecast']].copy()
submission.columns = ['Outlet_ID', 'Total_Latent_Demand']

# Round to 2 decimal places as per FMCG standards
submission['Total_Latent_Demand'] = submission['Total_Latent_Demand'].round(2)

print('Submission preview:')
print(submission.head())

# Ensure no missing outlets
master = pd.read_csv(ROOT / 'data' / 'silver' / 'outlet_master_clean.csv')
all_ids = master['Outlet_ID'].unique()
missing_ids = set(all_ids) - set(submission['Outlet_ID'])

if missing_ids:
    print(f'Warning: {len(missing_ids)} outlets missing from predictions. Filling with global mean.')
    mean_val = submission['Total_Latent_Demand'].mean()
    missing_df = pd.DataFrame({'Outlet_ID': list(missing_ids), 'Total_Latent_Demand': mean_val})
    submission = pd.concat([submission, missing_df], ignore_index=True)

# Final Save
team_name = 'SkyNet'
submission_file = OUTPUTS / f'{team_name}_predictions.csv'
submission.to_csv(submission_file, index=False)

print(f'Final submission file created: {submission_file}')
print(f'Total records in submission: {len(submission)}')

Results loaded: (10492, 58)
Submission preview:
   Outlet_ID  Total_Latent_Demand
0  OUT_00002               588.70
1  OUT_00003               559.17
2  OUT_00004               584.30
3  OUT_00007               571.77
4  OUT_00008               583.70
Final submission file created: C:\Users\User\Documents\Projects\Datastorm\SkyNet-datastorm-v7\outputs\SkyNet_predictions.csv
Total records in submission: 10492
